In [1]:
import os
os.chdir("../")

In [2]:
import os
import pandas as pd
import altair as alt
import glob
import re
from pathlib import Path

In [3]:
# Create figures directory
os.makedirs("figures", exist_ok=True)

alt.theme.enable('fivethirtyeight')

ThemeRegistry.enable('fivethirtyeight')

In [4]:
DATA_DIR = "data/train-modified/"

In [5]:
def get_target_language_name(filename):
    """
    Extracts target language from filenames.
    Handles variations like 'English-Assamese.csv' or 'English - Karbi.csv'
    """
    match = re.search(r'English\s*-\s*([a-zA-Z]+)', filename, re.IGNORECASE)
    if match:
        return match.group(1).strip().capitalize()
    return "Unknown"

def compute_statistics(file_path, category):
    filename = os.path.basename(file_path)
    tgt_lang = get_target_language_name(filename)
    
    # Load data: we know it's a CSV now.
    try:
        # Read the CSV. 
        # Using on_bad_lines='skip' just in case there are malformed rows in the raw text.
        df = pd.read_csv(file_path, on_bad_lines='skip')
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return None

    # Drop nulls
    df = df.dropna()
    
    # Standardize column names based on position since format is guaranteed
    # Col 0: English, Col 1: Target Language
    df.columns = ['English', 'Target']
    
    # Ensure all data is treated as strings
    df['English'] = df['English'].astype(str)
    df['Target'] = df['Target'].astype(str)
    
    # 1. Word Counts (using simple whitespace splitting)
    df['en_words'] = df['English'].apply(lambda x: len(x.split()))
    df['tgt_words'] = df['Target'].apply(lambda x: len(x.split()))
    
    # 2. Character Counts
    df['en_chars'] = df['English'].apply(len)
    df['tgt_chars'] = df['Target'].apply(len)
    
    # 3. Length Ratio (Target Words / Source Words)
    # Added a small epsilon (1e-5) to prevent division by zero for empty strings
    df['length_ratio'] = df['tgt_words'] / (df['en_words'] + 1e-5)
    
    # 4. Type-Token Ratio (TTR) - Measures lexical richness
    tgt_vocab = set(" ".join(df['Target'].tolist()).split())
    tgt_tokens = df['tgt_words'].sum()
    ttr = len(tgt_vocab) / (tgt_tokens + 1e-5) if tgt_tokens > 0 else 0

    stats = {
        'Language': tgt_lang,
        'Category': category,
        'Total_Sentences': len(df),
        'Avg_En_Words': df['en_words'].mean(),
        'Avg_Tgt_Words': df['tgt_words'].mean(),
        'Avg_En_Chars': df['en_chars'].mean(),
        'Avg_Tgt_Chars': df['tgt_chars'].mean(),
        'Avg_Length_Ratio': df['length_ratio'].mean(),
        'Type_Token_Ratio': ttr,
        # Sample up to 5000 rows for distribution plotting to keep memory/charts manageable
        'distribution_sample': df[['en_words', 'tgt_words', 'length_ratio']].sample(n=min(len(df), 5000)).assign(Language=tgt_lang)
    }
    return stats

In [8]:
all_stats = []
dist_dataframes = []

categories = ["Category I", "Category II"]

for category in categories:
    search_path = os.path.join(DATA_DIR, category, "*.csv")
    for file_path in glob.glob(search_path):
        print(f"Processing {os.path.basename(file_path)}...")
        stats = compute_statistics(file_path, category)
        if stats:
            dist_dataframes.append(stats.pop('distribution_sample'))
            all_stats.append(stats)

if not all_stats:
    print("No data found. Please check your DATA_DIR path.")
    exit()

Processing English-Mizo.csv...
Processing English-Meitei.csv...
Processing English-Khasi.csv...
Processing English-Nyishi.csv...
Processing English-Assamese.csv...
Processing English-Manipuri.csv...
Processing English-Bodo.csv...
Processing English-Nagamese.csv...
Processing English - Karbi.csv...
Processing English-Kokborok.csv...
Processing English-Tagin.csv...


In [9]:
summary_df = pd.DataFrame(all_stats)
dist_df = pd.concat(dist_dataframes, ignore_index=True)

print("\n--- Summary Statistics ---")
print(summary_df.to_string(index=False))


--- Summary Statistics ---
Language    Category  Total_Sentences  Avg_En_Words  Avg_Tgt_Words  Avg_En_Chars  Avg_Tgt_Chars  Avg_Length_Ratio  Type_Token_Ratio
    Mizo  Category I            49989     19.630619      20.887015     95.804397      97.677809          1.097256          0.036036
  Meitei  Category I            17000     18.422059      17.952824    112.254529     117.966294          1.000888          0.188124
   Khasi  Category I            26000     29.949577      36.494346    139.096538     156.132769          1.257194          0.008560
  Nyishi  Category I            60000      5.631450       5.397933     27.942117      30.250583          1.011162          0.120642
Assamese  Category I            54000     19.140593      16.268056     95.120148      91.288667          0.878787          0.077325
Manipuri  Category I            23687     17.837675      15.093680    102.790729     103.701566          0.912595          0.127010
    Bodo Category II            15214     15.000

# Observations

Bodo, Meitei, Manipuri, and Assamese are in their native scripts, while all other target languages (Mizo, Khasi, Nyishi, Nagamese, Karbi, Kokborok, Tagin) are in transliterated text (Latin script).

## 1. The Orthographic Divide & Tokenizer Fragmentation

Sharing a single vocabulary across distinct scripts (e.g., Devanagari, Bengali-Assamese, Meitei Mayek, and Latin) introduces a severe inefficiency.

**The Problem:** There is virtually zero subword overlap between the native script languages and the transliterated languages. If we train a standard Byte-Pair Encoding (BPE) model uniformly across this corpus, the massive transliterated datasets (like Nyishi's 60k sentences) will dominate the vocabulary slots with Latin subwords, starving the native scripts of representation.

**The Solution:** We cannot rely on a naive shared vocabulary. We must either force a balanced vocabulary allocation (e.g., explicitly reserving 30% of vocab slots for native scripts) or use a Byte-Level BPE (BBPE) or a SentencePiece model with byte_fallback=True to ensure the model can gracefully back off to individual Unicode bytes for rare native characters without producing <UNK> tokens. In contrast we shall transliterate everything and perform the training but it has to address the following:

    Lossy Phonetic Mapping: Standard English Latin characters lack the phonetic depth of Indic abugidas. Indic scripts perfectly
    distinguish between aspirated/unaspirated consonants (e.g., k vs. kh) and dental/retroflex consonants (e.g., a soft t vs. a hard T).
    Unless you use a highly complex transliteration scheme with special diacritics (like IAST), we will map distinct native words to the
    exact same Latin string, causing the model to lose semantic meaning.
    
    The Back-Transliteration Bottleneck: To participate in the evaluation and submit our results, our final output must be in the native
    scripts for Bodo, Meitei, Manipuri, and Assamese. This means your inference pipeline becomes:
    
    English Source $\rightarrow$ Model $\rightarrow$ Latin Output $\rightarrow$ Back-Transliterator Tool $\rightarrow$ Native Script Output
    
    If our model outputs a perfect translation, but your back-transliterator script makes a spelling error converting the Latin back to
    Meitei Mayek, your BLEU/ChrF scores will tank.



## 2. Interpreting TTR: Morphological Richness vs. Spelling Noise

The exceptionally high Type-Token Ratios (TTR) in Karbi (0.34) and Nagamese (0.23) suggested complex morphology. The transliteration context changes this.
**The Transliteration Factor:** When languages lacking a rigid, universally adopted Latin orthography are transliterated, translators often spell the exact same word in multiple different ways (e.g., phonetically spelling a sound as "sh" vs "s", or "oo" vs "u").

**The Implication:** The high TTR in transliterated Category II languages is likely a mixture of true lexical diversity and orthographic noise (spelling variations).

**The Solution:** Apply aggressive text normalization (lowercasing, standardizing phonetic Latin clusters) to the transliterated datasets before tokenizer training. During model training, consider applying character-level noise or subword regularization (BPE-dropout) to make the encoder robust to these spelling variations.

## 3. Script Constraints & Expansion Ratios

The ratio of Target words to English words (Avg_Length_Ratio) aligns strongly with the script being used.

**Native Script Contraction:** Languages in their native scripts (Assamese 0.87, Manipuri 0.91) generally exhibit contraction. Native Indic scripts are highly efficient at packing complex phonetics and postpositions into fewer space-separated word units (and fewer visual characters).

**Transliteration Expansion:** Transliterated languages (Khasi 1.25, Karbi 1.21) expand significantly. Representing Indic phonetics in Latin script often requires multiple characters (digraphs) and more frequent use of spaces, artificially increasing the word count.

**The Solution:** Your shared decoder generation scripts must account for this divide. A single max_length parameter will fail. Implement a dynamic length penalty during beam search that is conditioned on the target language/script ID.

## 4. Severe Corpus Imbalance (Sampling Strategy Required)

The dataset exhibits an extreme skew, ranging from 60,000 sentences (Nyishi) down to just 1,000 sentences (Karbi).

**Implication:** The high-resource transliterated languages will easily dominate the gradient updates, causing the model to catastrophically forget the low-resource native script languages (like Bodo).

**Action:** Implement temperature-based data sampling (e.g., $T=1.5$ or $T=5.0$) during batch creation to artificially upsample Category II languages and ensure balanced script representation throughout the training lifecycle.


In [11]:
print("\nGenerating charts...")

# 1. Corpus Size per Language
chart_size = alt.Chart(summary_df).mark_bar().encode(
    x=alt.X('Total_Sentences:Q', title='Number of Parallel Sentences'),
    y=alt.Y('Language:N', sort='-x', title='Target Language'),
    color=alt.Color('Category:N', scale=alt.Scale(scheme='tableau10')),
    tooltip=['Language', 'Total_Sentences', 'Category']
).properties(
    title='Parallel Corpus Size per Language',
    width=600, height=400
)
chart_size.save('figures/1_corpus_size.png', engine='vl-convert')

# 2. Lexical Diversity (Type-Token Ratio)
chart_ttr = alt.Chart(summary_df).mark_bar().encode(
    x=alt.X('Type_Token_Ratio:Q', title='Type-Token Ratio (Lexical Richness)'),
    y=alt.Y('Language:N', sort='-x', title='Target Language'),
    color=alt.Color('Category:N', scale=alt.Scale(scheme='set2')),
    tooltip=['Language', 'Type_Token_Ratio']
).properties(
    title='Lexical Diversity (TTR) across Languages',
    width=600, height=400
)
chart_ttr.save('figures/2_lexical_diversity_ttr.png', engine='vl-convert')

# 3. Source vs Target Sentence Length (Scatter Plot)
chart_len_scatter = alt.Chart(dist_df).mark_circle(opacity=0.3).encode(
    x=alt.X('en_words:Q', title='English Words per Sentence', scale=alt.Scale(domain=[0, 100], clamp=True)),
    y=alt.Y('tgt_words:Q', title='Target Words per Sentence', scale=alt.Scale(domain=[0, 100], clamp=True)),
    color=alt.Color('Language:N', legend=alt.Legend(title="Language")),
    tooltip=['Language', 'en_words', 'tgt_words']
).properties(
    title='English vs Target Sentence Length Correlation',
    width=600, height=500
)
# Reference line (y=x) to easily see if languages are consistently longer or shorter than English
line = alt.Chart(pd.DataFrame({'x': [0, 100], 'y': [0, 100]})).mark_line(color='red', strokeDash=[5,5]).encode(x='x', y='y')
(chart_len_scatter + line).save('figures/3_length_correlation.png', engine='vl-convert')

# 4. Length Ratio Distributions (Boxplot)
chart_ratio = alt.Chart(dist_df).mark_boxplot(extent='min-max').encode(
    x=alt.X('length_ratio:Q', title='Expansion Ratio (Target Words / English Words)', scale=alt.Scale(domain=[0, 4], clamp=True)),
    y=alt.Y('Language:N', title='Language'),
    color='Language:N'
).properties(
    title='Sentence Expansion Ratio Distribution',
    width=600, height=500
)
chart_ratio.save('figures/4_length_ratio_boxplot.png', engine='vl-convert')


Generating charts...
